In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
data_path = '/content/drive/MyDrive/cu_ai_grant/data'
metrics_path = '/content/drive/MyDrive/cu_ai_grant/metrics'
BASE = "Qwen/Qwen3-4B-Instruct-2507"

In [3]:
!pip -q install -U "transformers==4.55.2" "trl==0.19.1" "peft==0.16.0" bitsandbytes datasets scikit-learn

In [4]:
import json, torch, numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
import pickle, warnings
from scipy.sparse import hstack
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig
from peft import LoraConfig, PeftModel
warnings.filterwarnings("ignore")

set_seed(42)

# Основная часть кода

Считываем данные

In [5]:
def load_jsonl(p): return [json.loads(l) for l in open(p)]

kid_adult = load_jsonl(f"{data_path}/kid_adult.jsonl")
good_bad = load_jsonl(f"{data_path}/good_bad.jsonl")
test_style = load_jsonl(f"{data_path}/public_test_style.jsonl")
test_quality = load_jsonl(f"{data_path}/public_test_quality.jsonl")

d = pickle.load(open(f"{metrics_path}/style_clf.pkl", "rb"))
style_clf, style_vecs = d["clf"], d["vecs"]

In [34]:
def p_simple(texts):
    X = hstack([v.transform(texts) for v in style_vecs])
    return style_clf.predict_proba(X)[:, 1]

print(p_simple([r["kid"] for r in test_style]).mean(), p_simple([r["adult"] for r in test_style]).mean())

0.9747509734539157 0.01841233282530796


Первое задание

In [7]:
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(BASE)

In [38]:
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")

sft_data = Dataset.from_list([{"messages": [
    {"role": "user", "content": r["prompt"]},
    {"role": "assistant", "content": r["kid"]}]} for r in kid_adult])

# короче ранг 16 экспериментальным путем, большой слишком конечно я не ставил потому что ждать лень, alpha=2r база, слои по максимуму захватил
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

args = SFTConfig(output_dir="sft", num_train_epochs=1, per_device_train_batch_size=2,
                 gradient_accumulation_steps=8, learning_rate=2e-4, lr_scheduler_type="cosine",
                 warmup_ratio=0.03, max_length=1024, fp16=True, logging_steps=10,
                 report_to="none", seed=42)

trainer = SFTTrainer(model=model, args=args, train_dataset=sft_data,
                     processing_class=tok, peft_config=lora)
trainer.train()
trainer.save_model("sft")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Step,Training Loss
10,2.123000
20,1.440900
30,1.333800
40,1.222600
50,1.216600
60,1.178300
70,1.189800
80,1.170000
90,1.193100


In [41]:
def generate(model, prompts, max_new=256):
    model.eval()
    outs = []
    for p in prompts:
        ids = tok.apply_chat_template([{"role": "user", "content": p}],
                                      add_generation_prompt=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=max_new, do_sample=False)
        outs.append(tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True))
    return outs

gens = generate(trainer.model, [r["prompt"] for r in test_style])
m1 = float(p_simple(gens).mean())
print("mean P_simple =", round(m1, 4))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

mean P_simple = 0.9787


In [45]:
import gc
del trainer, model
gc.collect(); torch.cuda.empty_cache()

In [48]:
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")
model = PeftModel.from_pretrained(base, "sft", is_trainable=True) # подгружаем заново ту же модель чтобы была возможность подчистить память от тренера

dpo_data = Dataset.from_list([{
    "prompt": [{"role": "user", "content": r["prompt"]}],
    "chosen": [{"role": "assistant", "content": r["kid"]}],
    "rejected": [{"role": "assistant", "content": r["adult"]}]} for r in kid_adult])

args = DPOConfig(output_dir="dpo_style", num_train_epochs=1, per_device_train_batch_size=1,
                 gradient_accumulation_steps=16, learning_rate=1e-7, beta=0.1,
                 max_length=1024, max_prompt_length=512, fp16=True, logging_steps=10,
                 report_to="none", seed=42)

trainer = DPOTrainer(model=model, args=args, train_dataset=dpo_data, processing_class=tok)
trainer.train()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Extracting prompt in train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Step,Training Loss
10,0.186800
20,0.071700
30,0.141600
40,0.116300
50,0.121600
60,0.106900
70,0.112600
80,0.284000
90,0.348600


TrainOutput(global_step=94, training_loss=0.17001754869806004, metrics={'train_runtime': 1488.9332, 'train_samples_per_second': 1.0, 'train_steps_per_second': 0.063, 'total_flos': 0.0, 'train_loss': 0.17001754869806004, 'epoch': 1.0})

In [50]:
gens = generate(trainer.model, [r["prompt"] for r in test_style])
m2 = float(p_simple(gens).mean())
print("Task 2: mean P_simple =", round(m2, 4))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

Task 2: mean P_simple = 0.9798


Видим небольшой, на тысячные, но всё-таки рост

In [54]:
trainer.save_model("dpo_style")

In [55]:
del trainer, model, base
gc.collect(); torch.cuda.empty_cache()

In [8]:
from trl import RewardTrainer, RewardConfig
from transformers import AutoModelForSequenceClassification

rm = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=1,
                                                        quantization_config=bnb, device_map="auto")
rm.config.pad_token_id = tok.pad_token_id

rm_data = Dataset.from_list([{
    "chosen": [{"role": "user", "content": r["instruction"]},
               {"role": "assistant", "content": r["chosen"]}],
    "rejected": [{"role": "user", "content": r["instruction"]},
                 {"role": "assistant", "content": r["rejected"]}]} for r in good_bad])

lora_rm = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, task_type="SEQ_CLS",
                     target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

args = RewardConfig(output_dir="rm", num_train_epochs=1, per_device_train_batch_size=1,
                    gradient_accumulation_steps=16, learning_rate=1e-5, max_length=1024,
                    fp16=True, logging_steps=10, report_to="none", seed=42)

trainer = RewardTrainer(model=rm, args=args, train_dataset=rm_data,
                        processing_class=tok, peft_config=lora_rm)
trainer.train()
trainer.save_model("rm")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-4B-Instruct-2507 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/2226 [00:00<?, ? examples/s]

Map:   0%|          | 0/2226 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2226 [00:00<?, ? examples/s]

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
10,0.623200
20,0.359900
30,0.223000
40,0.152200
50,0.088500
60,0.052500
70,0.064000
80,0.041400
90,0.069700
100,0.071300


In [9]:
def rm_score(model, prompt, answer):
    ids = tok.apply_chat_template([{"role": "user", "content": prompt},
                                   {"role": "assistant", "content": answer}],
                                  return_tensors="pt").to(model.device)
    with torch.no_grad():
        return model(ids).logits[0, 0].item()

rm_model = trainer.model
rm_model.eval()
acc = np.mean([rm_score(rm_model, r["prompt"], r["chosen"]) > rm_score(rm_model, r["prompt"], r["rejected"])
               for r in test_quality])
print("Pairwise accuracy =", acc)

Task 3: pairwise accuracy = 1.0


In [15]:
!git add ai_red.ipynb

Everything up-to-date
